# System D: Qwen2.5-1.5B-Instruct + LoRA (Phase 3)

Fine-tunes LoRA adapters (r=16, alpha=32, on the attention projections) on the 3,382 training disses. The base model
already knows English, so there is no Stage 1. The prompt carries the same information as System A's prefix (traits,
intensity, the four planned end words) and the answer is four lines, each tagged with its end word first:

```
[late] you live by your clock, not a minute too late,
[wait] your life's all in stock tips and nothing to wait.
```

The tag makes the model commit to the rhyme word before it writes the line; it is stripped on decoding (see
`generation/qwen_lora.py`). Nothing forces the end word as in System A, so the report measures how often it lands.
The last section compares D with System A (`gen_A_rev_stage2_best.pt`) on the same prompts and sampling settings.

**On Colab:** Runtime → Change runtime type → **T4 GPU**, then Run all. The repo is cloned from GitHub and the adapter
is saved to Google Drive (`MyDrive/CS425/checkpoints/gen_D_1.5b_best/`). If the runtime disconnects, run all again:
training resumes from the last finished epoch.

**Expected time on a T4** (about 410k tokens per epoch): setup and model download 3–5 min, training 3 epochs about
15–25 min, sampling 80 prompts × 4 and the test loss about 5–8 min, comparison with A about 1 min. **About 25–40 min
in all.** The training log prints the time left in each epoch.

**Other sizes:** change `MODEL_NAME` (e.g. `Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen2.5-3B-Instruct`); the checkpoint
names follow it (`gen_D_0.5b`, `gen_D_3b`). For 3B on a T4, set `BATCH_SIZE, ACCUM = 4, 4`.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/SchoolWork-awork2103/CS425-Project.git"
BRANCH = "feat/phase0"
DRIVE_CKPT_DIR = "/content/drive/MyDrive/CS425/checkpoints"
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # a few batches only, to check the notebook runs end to end

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    root = "/content/CS425-Project"
    if os.path.exists(root):
        subprocess.run(["git", "-C", root, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, root], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "transformers>=4.56", "sentencepiece",
                    "pyyaml", "pronouncing", "wordfreq"], check=True)
    os.chdir(root)
    CKPT_DIR = DRIVE_CKPT_DIR
else:
    if os.path.basename(os.getcwd()) == "generation":
        os.chdir("..")
    CKPT_DIR = "checkpoints"
BASELINE_A = os.path.join(CKPT_DIR, "gen_A_rev_stage2_best.pt")   # System A (final run), for the comparison
if SMOKE_TEST:
    CKPT_DIR = os.path.join(CKPT_DIR, "smoke_test")
sys.path[:0] = [os.getcwd(), os.path.join(os.getcwd(), "generation")]
print("project:", os.getcwd(), "| checkpoints:", CKPT_DIR, "| smoke test:", SMOKE_TEST)

In [ ]:
import json, math, random, re, time
import torch
from transformers import AutoTokenizer
from utils import load_config, project_path, set_seed
from sequences import read_jsonl
from qwen_lora import load_base, add_lora, example, make_collate, train, run_epoch, generate, report

cfg = load_config()
SEED = set_seed(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = os.environ.get("QWEN_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
NAME = "gen_D_" + re.search(r"(\d+(?:\.\d+)?)B", MODEL_NAME).group(1) + "b"

diss_dir = project_path(cfg["paths"]["diss_dir"])
rows = {s: read_jsonl(diss_dir / f"{s}.jsonl") for s in ("train", "val", "test")}
trait_info = json.load(open(project_path(cfg["paths"]["traits"])))
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
examples = {s: [example(tok, r) for r in rows[s]] for s in ("train", "val")}

lens = sorted(len(e["ids"]) for e in examples["train"])
print("device:", device, torch.cuda.get_device_name() if device == "cuda" else "", "| model:", MODEL_NAME,
      "| checkpoints:", NAME)
print({s: len(v) for s, v in rows.items()}, f"| tokens per example: median {lens[len(lens) // 2]}, max {lens[-1]}, "
      f"{sum(lens):,} per epoch")
print(tok.decode(examples["train"][0]["ids"]))

In [ ]:
# Hyperparameters (pipeline.md, System D)
LORA = dict(r=16, alpha=32, dropout=0.05)    # on the attention projections q, k, v, o
LR = 2e-4
EPOCHS = 3
BATCH_SIZE, ACCUM = 8, 2                     # 16 examples per optimizer step; for 3B on a T4 use 4, 4
SAMPLING = dict(temperature=0.8, top_p=0.9, rep_penalty=1.2)    # same as System A
N_SAMPLES, K_PROMPTS = 4, 40                 # samples per prompt, prompts per set (as in train_gru.ipynb)

MAX_BATCHES = 3 if SMOKE_TEST else None
if SMOKE_TEST:
    EPOCHS, BATCH_SIZE, N_SAMPLES, K_PROMPTS = 1, 2, 2, 2

## Training

In [ ]:
t = time.time()
model, _ = load_base(MODEL_NAME, device)
model = add_lora(model, **LORA)
model.print_trainable_parameters()
print(f"model ready in {time.time() - t:.0f}s")
hist = train(model, tok, examples["train"], examples["val"], device, CKPT_DIR, NAME, lr=LR, epochs=EPOCHS,
             batch_size=BATCH_SIZE, accum=ACCUM, seed=SEED, max_batches=MAX_BATCHES)
if device == "cuda":
    print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

## Samples
Validation disses (training traits) and held-out traits from the test set: the same prompts and sampling settings
as `train_gru.ipynb`.

In [ ]:
val_rows = list(rows["val"])
random.Random(SEED).shuffle(val_rows)
heldout_rows = [r for r in rows["test"] if r["kind"] == "heldout"]
random.Random(SEED).shuffle(heldout_rows)
to_prompt = lambda r: {"traits": r["traits"], "intensity": r["intensity"], "end_words": r["end_words"]}
prompt_sets = {"seen": [to_prompt(r) for r in val_rows[:K_PROMPTS]],
               "heldout": [to_prompt(r) for r in heldout_rows[:K_PROMPTS]]}

def sample_d(prompts):
    torch.manual_seed(SEED)
    return [[s["lines"] for s in generate(model, tok, p["traits"], p["intensity"], p["end_words"], n=N_SAMPLES,
                                          device=device, **SAMPLING)] for p in prompts]

t = time.time()
poems_d = {k: sample_d(v) for k, v in prompt_sets.items()}
print(f"sampled {sum(len(v) for v in prompt_sets.values()) * N_SAMPLES} poems in {(time.time() - t) / 60:.1f} min")
reports_d = {}
for k, title in (("seen", "training traits (validation disses)"), ("heldout", "held-out traits (never seen in training)")):
    print(f"\n=== {title}")
    reports_d[k] = report(prompt_sets[k], poems_d[k], trait_info, show=6)

In [ ]:
# Test loss on the diss test set, separately for training traits and held-out traits. Perplexity is per Qwen token,
# so it is not comparable with System A's (different tokenizer).
collate = make_collate(tok.pad_token_id)
def test_loss(rs):
    dl = torch.utils.data.DataLoader([example(tok, r) for r in rs], BATCH_SIZE * 2, collate_fn=collate)
    return run_epoch(model, dl, device, max_batches=MAX_BATCHES)

test = {"seen_traits": test_loss([r for r in rows["test"] if r["kind"] != "heldout"]),
        "heldout_traits": test_loss(heldout_rows)}
for name, loss in test.items():
    print(f"test {name:15s} loss {loss:.4f}  perplexity {math.exp(loss):6.2f}")

## Comparison with System A
Same prompts, same sampling settings, same checks. Needs `gen_A_rev_stage2_best.pt` in the checkpoint folder.

In [ ]:
reports_a = None
if os.path.exists(BASELINE_A):
    from gru_lm import load_generator as load_gru, generate as generate_gru
    gru, codec = load_gru(BASELINE_A, cfg, device)
    torch.manual_seed(SEED)
    poems_a = {k: [[s["lines"] for s in generate_gru(gru, codec, p["traits"], p["intensity"], p["end_words"],
                                                     n=N_SAMPLES, device=device, **SAMPLING)] for p in v]
               for k, v in prompt_sets.items()}
    reports_a = {k: report(prompt_sets[k], poems_a[k], trait_info, log=lambda *a: None) for k in prompt_sets}

    keys = ("four_lines", "end_word_hit", "couplets_rhyme", "aabb", "syllables", "lexical", "trait_mention", "distinct_2")
    print(f"{'':16s}{'A seen':>10s}{'D seen':>10s}{'A held':>10s}{'D held':>10s}")
    for key in keys:
        print(f"{key:16s}" + "".join(f"{r[s][key]:>10.1%}" for s in ("seen", "heldout") for r in (reports_a, reports_d)))

    for i, p in enumerate(prompt_sets["seen"][:3]):
        print(f"\n[{', '.join(p['traits']) or 'no traits'} | {p['intensity']} | rhymes {' '.join(p['end_words'])}]")
        for label, poems in (("A", poems_a), ("D", poems_d)):
            print(f"  {label}: " + "\n     ".join(poems["seen"][i][0]))
else:
    print("no System A checkpoint at", BASELINE_A)

In [ ]:
summary = {"name": NAME, "model": MODEL_NAME, "lora": LORA, "lr": LR, "epochs": EPOCHS,
           "batch_size": BATCH_SIZE * ACCUM, "sampling": SAMPLING, "history": hist,
           "best_val_loss": min(h["val_loss"] for h in hist), "test_loss": test,
           "samples": reports_d, "comparison_A": reports_a}
json.dump(summary, open(os.path.join(CKPT_DIR, f"{NAME}_summary.json"), "w"), indent=2)
print("saved", os.path.join(CKPT_DIR, f"{NAME}_summary.json"))